# Noise assessments from LAT maps 

We use pre-computed noise spectra per OT/band to fit power law noise profiles and compare to noise spectra forecasts ([SO: Science goals and forecasts
](https://arxiv.org/abs/1808.07445)). We also derive an NET estimate from the maps white noise level and compare with estimates from TOD statistics and commissionning PSD estimates.

## Imports

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from pspy import so_spectra, pspy_utils
import camb
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from pixell import enmap, enplot
import os
import h5py

## Theory spectra

In [ ]:
def make_theory_cls(cosmo_params, output_dir, ell_min=2, ell_max=10000):
    
    # Define range of angular scales
    _ell = np.arange(ell_min, ell_max)
    
    # The base cosmology model is set with these params
    pars = camb.set_params(**cosmo_params)
    
    # Set how far in multipole we want the power spectra, 
    # and turn on defaults for the params.
    pars.set_for_lmax(ell_max, lens_potential_accuracy=1)
    
    # Calculate results for these parameters 
    results = camb.get_results(pars)
    
    # Get dictionary of CAMB power spectra
    powers = results.get_cmb_power_spectra(pars, CMB_unit="muK")
    
    # Write to file
    os.makedirs(output_dir, exist_ok=True)
    cl_file = os.path.join(output_dir, "cl_camb.dat")
    np.savetxt(cl_file, np.hstack([_ell[:, np.newaxis], powers["total"][ell_min:ell_max]]))
    
    return

In [ ]:
# Set up cosmological parameters
cosmo_params = {
        "H0": 67.36,
        "As": 1e-10 * np.exp(3.044),
        "ombh2": 0.02237,
        "omch2": 0.1200,
        "ns": 0.9649,
        "Alens": 1.0,
        "tau": 0.0544,
    }

# Compute theory spectra
outdir = "power_spectra/"
make_theory_cls(cosmo_params, output_dir = outdir)

# Convert to D_ell and write to file
ell_max = 8000
ell_theory, ps_theory = pspy_utils.ps_lensed_theory_to_dict(f"{outdir}cl_camb.dat", "Cl", lmax=ell_max)

## Load noise spectra

In [ ]:
path = "/scratch/gpfs/SIMONSOBS/users/he3297/SO_analyses/LAT/spectra_0925" # replace with path to your spectra
spectra = ["TT", "TE", "TB", "ET", "BT", "EE", "EB", "BE", "BB"]

In [ ]:
# Initialize list of OT/bands
bands = ["f090", "f150", "f220", "f280"]
tubes = {}
tubes["f090"] = ["i1", "i3", "i4", "i6"]
tubes["f150"] = ["i1", "i3", "i4", "i6"]
tubes["f220"] = ["i5", "c1"]
tubes["f280"] = ["i5", "c1"]

In [ ]:
# Load noise spectra
Nl = {}
for b in bands:
    for t in tubes[b]:
        lbb, Nl[f'{t}_{b}'] = so_spectra.read_ps(path+f'/Dl_SO_{t}_{b}xSO_{t}_{b}_noise.dat', spectra=spectra)

Loaded spectra are of the form $\ell(\ell+1)C_\ell/2\pi$ with Gaussian beams deconvolved let's undo this

In [ ]:
# Beams
fwhm_f090 = 2.2 # arcmin 
fwhm_f150= 1.5
fwhm_f220= 0.96
fwhm_f280= 0.8

# Compute gaussian beams
lmax = 8000
bl_f090 = hp.gauss_beam(fwhm=np.radians(fwhm_f090/60), pol=True, lmax=lmax)
bl_f150 = hp.gauss_beam(fwhm=np.radians(fwhm_f150/60), pol=True, lmax=lmax)
bl_f220 = hp.gauss_beam(fwhm=np.radians(fwhm_f220/60), pol=True, lmax=lmax)
bl_f280 = hp.gauss_beam(fwhm=np.radians(fwhm_f280/60), pol=True, lmax=lmax)

# Interpolate to the spectra ell bins
beams = {}
for b in bands:
    beams[b] = {}
    for i,s in enumerate(['T','E','B']):
        interp_func = interp1d(np.arange(lmax+1), eval(f'bl_{b}[:,{i}]'), bounds_error=False, fill_value=0.0)
        beams[b][s] = interp_func(lbb)

In [ ]:
# Get Nls
for b in bands:
    for t in tubes[b]:
        for s1 in ['T','E','B']:
            for s2 in ['T','E','B']:
                Nl[f'{t}_{b}'][s1+s2] *=  1 / (lbb*(lbb+1)) * 2*np.pi * beams[b][s1] * beams[b][s2]

## Pull some TOD stats & survey parameters

In [ ]:
nsplits = 2

In [ ]:
# Let's get npix & fsky for each tube/band/split
div = {}
survey_mask = {}
npix = {}
fsky = {}
for b in bands:
    for t in tubes[b]:
        for i in np.arange(nsplits):
            div[f'{t}_{b}_{i}'] = enmap.read_map(f'deep56_4/deep56_{t}_2way_{i}_{b}_sky_ivar.fits')
            survey_mask[f'{t}_{b}_{i}'] = div[f'{t}_{b}_{i}']>0
            npix[f'{t}_{b}_{i}'] = np.sum(survey_mask[f'{t}_{b}_{i}'])
            Npix_tot = 4*np.pi / enmap.pixsize(survey_mask[f'{t}_{b}_{i}'].shape, survey_mask[f'{t}_{b}_{i}'].wcs)
            fsky[f'{t}_{b}_{i}'] = npix[f'{t}_{b}_{i}'] / Npix_tot
            f = fsky[f'{t}_{b}_{i}']
            print(f'{t} {b} {i}/2: fsky = {f*100:.1f}%')

In [ ]:
# These are roughly similar let's use a single value
fsky_data = sum(fsky.values()) /len(fsky)
print(f'fsky = {fsky_data*100:.1f}%')

In [ ]:
# Let's get integration times and tubes sensitivities from TOD stats dump by Sigurd
integration_times = {}
sens = {}
ubands = {}
for b in bands:
    for t in tubes[b]:
        for i in np.arange(nsplits):
            with h5py.File(f'deep56_info/deep56_{t}_2way_{i}_info.hdf', 'r') as f:
                # Access specific fields
                dataset = f['obstab']
                ubands[f'{t}'] = f['ubands'][:]
    
                # Read arrays of interest
                data = dataset[:]
                integration_times[f'{t}_{i}']  = np.sum(data['dur'])
                sens[f'{t}_{i}'] = data['sens']
        integration_times[f'{t}'] = 0
        sens[f'{t}'] = []
        for i in np.arange(nsplits):
            integration_times[f'{t}'] += integration_times[f'{t}_{i}']
            sens[f'{t}'].append(sens[f'{t}_{i}'])
            integration_times.pop(f'{t}_{i}', None)
            sens.pop(f'{t}_{i}', None)
        sens[f'{t}'] = np.concatenate(sens[f'{t}'])

In [ ]:
# Print total integration time for each tube
tlist = set(x for v in tubes.values() for x in v)
time_per_tube = []
for t in tlist:
    time = integration_times[f'{t}']
    print(f'{t} Total integration time = { time/ 3600:.0f} hours')
    time_per_tube.append(time)

In [ ]:
# Except for c1, they all seem to have roughly the same duration let's pick the median 
duration = np.median(time_per_tube)

## Fit power law & compare to forecasts

In [ ]:
def power_law_model(log_l, alpha, log_lknee, Nwhite, Nred):
    """Power-law model in log space."""
    return np.log10(Nwhite  + Nred * (10**log_l / 10**log_lknee) ** alpha)

In [ ]:
ell = 1+np.arange(8000)

# ell range of fit
ell_min = 500
ell_max = 7000
mask_lbb = np.logical_and(lbb>ell_min, lbb<ell_max)
log_l = np.log10(lbb[mask_lbb])

In [ ]:
# Fit power laws to the noise spectra
Nl_fits = {}
for s in ["TT", "EE", "BB"]:
    Nl_fits[s] = {}
    if s == "TT":
        fixed_lknee = 1000 # Following forecast paper
    else:
        fixed_lknee = 700
    model = lambda log_l, alpha, Nwhite, Nred: power_law_model(log_l, alpha, np.log10(fixed_lknee), Nwhite, Nred)
    for b in bands:
        for t in tubes[b]:
            data = np.log10(Nl[f'{t}_{b}'][s][mask_lbb])
            popt, _ = curve_fit(model, log_l, data, p0 = [-3, 1e-4, 1e-4])
            alpha, Nwhite, Nred = popt
            Nl_fits[s][f'{t}_{b}'] = {}
            Nl_fits[s][f'{t}_{b}']['alpha'] = alpha
            Nl_fits[s][f'{t}_{b}']['Nwhite'] = Nwhite
            Nl_fits[s][f'{t}_{b}']['Nred'] = Nred

In [ ]:
# Global parameters used in SO forecast paper
fsky = 0.4
survey_efficiency = 0.2 * 0.85
survey_time = 5 * 365.25 * 86400. * survey_efficiency

# Some useful unit conversion utilities
def ukarcmin_to_uk2str(Nin):
    return Nin**2 / (60*180/np.pi)**2

def uk2str_to_ukarcmin(Nin):
    return np.sqrt(Nin * (60*180/np.pi)**2)

def uk2s_to_uk2str(Nin, fksy, survey_time):
    A = fsky * 4*np.pi
    factor = A / survey_time
    return Nin * factor

def uk2str_to_uk2s(Nin, fsky, survey_time):
    A = fsky * 4*np.pi
    factor = A / survey_time
    return Nin / factor

In [ ]:
# Noise spectra forecast
Nwhite_forecasts = {}
Nwhite_forecasts['goal'] = np.array([5.8, 6.3, 15, 37]) # in uK-arcmin
Nwhite_forecasts['baseline'] = np.array([8.0, 10.0, 22, 54]) # in uK-arcmin
# Convert to uk^2.str
Nwhite_forecasts['goal'] = ukarcmin_to_uk2str(Nwhite_forecasts['goal'])
Nwhite_forecasts['baseline'] = ukarcmin_to_uk2str(Nwhite_forecasts['baseline'])

Nred_forecasts = np.array([230.0, 1500.0, 17000.0, 31000.0]) # in uK^2.s
# Convert to uK^2.str
Nred_forecasts = uk2s_to_uk2str(Nred_forecasts, fsky, survey_time)

Nl_forecasts_baseline = {}
Nl_forecasts_goal = {}
for s in ["TT", "EE", "BB"]:
    Nl_forecasts_baseline[s] = {}
    Nl_forecasts_goal[s] = {}
    if s == "TT":
        fixed_lknee = 1000 # Following forecast paper
        alpha = -3.5
    else:
        fixed_lknee = 700
        alpha = -1.4
    for i,b in enumerate(bands):
        if s == "TT":
            Nl_forecasts_baseline[s][b] = 10 ** power_law_model(np.log10(ell), alpha, np.log10(fixed_lknee), Nwhite_forecasts['baseline'][i], Nred_forecasts[i])
            Nl_forecasts_goal[s][b] = 10 ** power_law_model(np.log10(ell), alpha, np.log10(fixed_lknee), Nwhite_forecasts['goal'][i], Nred_forecasts[i])
        else:
            Nl_forecasts_baseline[s][b] = 10 ** power_law_model(np.log10(ell), alpha, np.log10(fixed_lknee), 2*Nwhite_forecasts['baseline'][i], 2*Nwhite_forecasts['baseline'][i])
            Nl_forecasts_goal[s][b] = 10 ** power_law_model(np.log10(ell), alpha, np.log10(fixed_lknee), 2*Nwhite_forecasts['goal'][i], 2*Nwhite_forecasts['goal'][i])

In [ ]:
# Plot full survey noise spectra forecast
mask_ell = np.logical_and(ell > 0, ell < ell_max)
cmap = plt.get_cmap('viridis')
vmin, vmax = 0.0, 1.0  # Can customize the used dynamic range of the colormap
colors = [cmap(v) for v in np.linspace(vmin, vmax, len(bands))]

from matplotlib.lines import Line2D

for s in ["TT", "EE"]:
    plt.figure(figsize=[9, 7], dpi=200)
    plt.title(f'SO LAT {s} Noise Power Spectra Forecast ($f_{{sky}}$ = {fsky:.2f})', fontsize=16)

    # Plot forecasts
    for b, c in zip(bands, colors):
        plt.loglog(ell, Nl_forecasts_baseline[s][b], color=c, ls='-', lw=1.5)
        plt.loglog(ell, Nl_forecasts_goal[s][b], color=c, ls='--', lw=1.5)

    # Plot theory curve (Lensed CMB)
    cmb_line, = plt.plot(ell_theory, ps_theory[s], color='black', ls='-', lw=2.0, label=f'Lensed CMB {s}')

    # Axes setup
    plt.xlim(200, 8000)
    if s == "TT":
        plt.ylim(1e-6, 100)
    else:
        plt.ylim(1e-6, 1e-2)

    plt.xlabel(r'Multipole $\ell$', fontsize=14)
    plt.ylabel(r'$N_\ell~[\mathrm{\mu K}^2]$', fontsize=14)

    # --- Custom Legend Handles ---
    baseline_line = Line2D([0], [0], color='black', lw=1.5, ls='-', label='Baseline')
    goal_line = Line2D([0], [0], color='black', lw=1.5, ls='--', label='Goal')
    band_lines = [Line2D([0], [0], color=c, lw=2, ls='-', label=b) for b, c in zip(bands, colors)]

    legend_handles = [baseline_line, goal_line, *band_lines, cmb_line]

    plt.legend(handles=legend_handles, fontsize=12, loc="upper right")

In [ ]:
# Plot noise spectra along with rescaled forecasts
mask_ell = np.logical_and(ell>ell_min, ell<ell_max)
cmaps = ['Reds', 'Greens', 'Blues', 'Purples']
ntubes = {}
for b in bands:
    ntubes[b] = len(tubes[b])
# Derive scaling factors to get noise levels per tube expected for our maps 
scaling = {}
for b in bands:
    scaling[b] = survey_time / duration * ntubes[b] * fsky_data / fsky

for s in ["TT","EE","BB"]:
    
    plt.figure(figsize=[9,7], dpi=200)
    plt.title(f'{s} noise', fontsize=16)
    
    if s == "TT":
        fixed_lknee = 1000 # Following forecast paper
    else:
        fixed_lknee = 700
    model = lambda log_l, alpha, Nwhite, Nred: power_law_model(log_l, alpha, np.log10(fixed_lknee), Nwhite, Nred)
    
    for ib, b,cm in zip(np.arange(len(bands)),bands,cmaps):
        # Prepare colors
        cmap = plt.get_cmap(cm)
        vmin, vmax = 0.3, 1.0   # only use the top 70% of the colormap to avoid white dots
        colors = [cmap(v) for v in np.linspace(vmin, vmax, len(tubes[b]))]
        if s =="TT":
            alpha_forecast = -3.5
            Nw_g =  uk2str_to_ukarcmin(scaling[b] * Nwhite_forecasts['goal'][ib])
            Nw_b =  uk2str_to_ukarcmin(scaling[b] * Nwhite_forecasts['baseline'][ib])
            Nr_g =  uk2str_to_uk2s(scaling [b] * Nred_forecasts[ib], fsky_data, duration)
            Nr_b = Nr_g
        else:
            alpha_forecast = -1.4
            Nw_g =  uk2str_to_ukarcmin(2 * scaling[b] * Nwhite_forecasts['goal'][ib])
            Nw_b =  uk2str_to_ukarcmin(2 * scaling[b] * Nwhite_forecasts['baseline'][ib])
            Nr_g =  uk2str_to_uk2s(2 * scaling [b] * Nwhite_forecasts['goal'][ib], fsky_data, duration )
            Nr_b = uk2str_to_uk2s(2 * scaling [b] * Nwhite_forecasts['baseline'][ib], fsky_data, duration )
        plt.loglog(ell, scaling[b] * Nl_forecasts_goal[s][b], color=colors[-1], ls='--', lw=3.0, label=rf'{b} tubes Goal forecast: $\alpha$ = {alpha_forecast:.1f}, Nwhite = {Nw_g:.0f}$~\mu K$-arcmin, Nred = {Nr_g:.0f}$~\mu K^2.s$')
        plt.loglog(ell, scaling[b] * Nl_forecasts_baseline[s][b], color=colors[-1], ls='-', lw=3.0, label=rf'{b} tubes Baseline forecast: $\alpha$ = {alpha_forecast:.1f}, Nwhite = {Nw_b:.0f}$~\mu K$-arcmin, Nred = {Nr_b:.0f}$~\mu K^2.s$')
        for c,t in zip(colors,tubes[b]):
            # Best-fit
            alpha, Nwhite, Nred = Nl_fits[s][f'{t}_{b}'].values()
            Nw = uk2str_to_ukarcmin(Nwhite)
            Nr = uk2str_to_uk2s(Nred, fsky_data, duration)
            log_noisefit = model(np.log10(ell), *Nl_fits[s][f'{t}_{b}'].values())
            # Plot
            plt.loglog(lbb, Nl[f'{t}_{b}'][s], marker='o', color=c, ms= 6.0, mec='black', alpha=0.7, ls='', lw=1.0, label=f'{t} {b} noise bins')
            # plt.loglog(ell[mask_ell], scale * Nl_forecasts_goal[s][b][mask_ell], color=c, ls='-', lw=1.5, label=rf'Baseline forecast')
            plt.loglog(ell[mask_ell], 10**log_noisefit[mask_ell], color=c, ls='--', lw=1.5, label=rf'Best-fit {t} {b} noise power law: $\alpha$ = {alpha:.2f}, Nwhite = {Nw:.0f}$~\mu K$-arcmin, Nred = {Nr:.0f}$~\mu K^2.s$')
            plt.plot(ell_theory, ps_theory[s], color='black', ls='-', lw=2.0)
    plt.xlim(200, 7000)
    if s=="TT":
        plt.ylim(1e-5, 100)
    else:
        plt.ylim(1e-5, 1)
    plt.axvspan(xmin=0, xmax=ell_min, color='gray', alpha=0.2)
    plt.xlabel(r'Multipole $\ell$', fontsize=14)
    plt.ylabel(r'$N_\ell~[\mathrm{\mu K}^2]$', fontsize=14)
    plt.legend(fontsize=12, loc="center left", bbox_to_anchor=(1., 0.5)) 

## NET estimates

We compute NET estimates for each OT/band from our white noise fits in the noise spectra and compare to the distribution of NETs per obs. provided by Sigurd.

In [ ]:
# First let's reorganize the TOD based NET estimates and get medians and error
NETs_tods = {}
median_NET_tods = {}
err_NET_tods = {}
for t in tlist:
    for i,ub in enumerate(ubands[t]):
        b = ub.decode('utf-8')
        NETs_tods[f'{t}_{b}'] = sens[f'{t}'][:,i]
        median_NET_tods[f'{t}_{b}'] = np.median(NETs_tods[f'{t}_{b}'])
        err_NET_tods[f'{t}_{b}'] = np.std(NETs_tods[f'{t}_{b}'], ddof=1) / np.sqrt(NETs_tods[f'{t}_{b}'].size)

In [ ]:
# Estimating NETs from noise spectra
NETs_spectra = {}
for b in bands:
    for t in tubes[b]:
        # Get white noise level in uK^2.str
        Nw = Nl_fits['TT'][f'{t}_{b}']['Nwhite']
        # Convert to RMS per pixel
        pixarea = enmap.pixsize(survey_mask[f'{t}_{b}_0'].shape, survey_mask[f'{t}_{b}_0'].wcs)
        Nw = np.sqrt(Nw / pixarea)
        # Get average integration time per pixel
        tpix = 0
        for i in np.arange(nsplits):
            tpix += integration_times[f'{t}_{i}'] / npix[f'{t}_{b}_{i}']
        NETs_spectra[f'{t}_{b}'] = Nw * np.sqrt(tpix)

In [ ]:
# Plot per OT/band NET estimates and compare
for b in bands:
    ntubes = len(tubes[b])
    nx,ny = min(2,ntubes), ntubes // 2
    plt.figure(figsize=[8 * nx,6 * ny], dpi=200)
    plt.suptitle(f'{b} NETs', fontweight='bold', fontsize=18)
    for it,t in enumerate(tubes[b]):
        plt.subplot(eval(str(ny)+str(nx)+str(it+1)))
        total_duration = integration_times[f'{t}'] / 3600
        plt.title(f'{t}, total duration ={total_duration:.0f} hours', fontsize=16)
        NET = NETs_spectra[f'{t}_{b}']
        plt.axvline(NET, color='orange', ls='--', label=rf'from $N_\ell$: {NET:.1f} $\mu K \sqrt{{s}}$')
        plt.hist(NETs_tods[f'{t}_{b}'], bins=80, edgecolor='black', alpha=0.5)
        median = median_NET_tods[f'{t}_{b}']
        error = err_NET_tods[f'{t}_{b}']
        plt.axvline(median, color='darkblue', ls='--', label=f'median = {median:.1f} $\pm$ {error:.1f} $\mu K \sqrt{{s}}$')
        plt.axvspan(xmin=median - error, xmax=median + error, color='darkblue', alpha=0.2)
        plt.xlabel(r'NET $[\mu K \sqrt{s}]$', fontsize=12)
        plt.ylabel('Counts', fontsize=12)
        plt.xlim(median - 3*error*20,median + 3*error*20)
        plt.legend(loc = "upper right", fontsize=12)

### Let's get final results per band

In [ ]:
# Coadd all tube estimates to get a final estimate per band
band_NETs_spectra = {}
band_NETs_tods = {}
for b in bands:
    band_NETs_spectra[b] = 0
    band_NETs_tods[b] = 0
    for t in tubes[b]:
        band_NETs_spectra[b] += 1/NETs_spectra[f'{t}_{b}']**2
        band_NETs_tods[b] += 1/median_NET_tods[f'{t}_{b}']**2
    band_NETs_spectra[b] = np.sqrt(1/band_NETs_spectra[b])
    band_NETs_tods[b] = np.sqrt(1/band_NETs_tods[b])

In [ ]:
# Add some other references to compare to
NET_commissioning = {'f090':3.4 ,'f150':4.2, 'f220':11, 'f280':28.5} # Provided by Jack OS
NET_baseline = {'f090':5.4 ,'f150':6.7, 'f220':15, 'f280':36}
NET_goal = {'f090':3.9 ,'f150':4.2, 'f220':10, 'f280':25}

In [ ]:
# Print results
for b in bands:
    print(f'{b}: NET from Nl = {band_NETs_spectra[b]:.1f}uK.√s \t NET from tod stats. = {band_NETs_tods[b]:.1f}uK.√s \t Baseline forecast = {NET_baseline[b]:.1f}uK.√s \t Goal forecast = {NET_goal[b]:.1f}uK.√s')